# Iceberg Metadata Tables

Apache Iceberg exposes its internal metadata as queryable SQL tables. Unlike Hive's one-off `SHOW PARTITIONS`, these are **full SQL tables** — you can sort, aggregate, join, and time-travel against them.

This notebook covers every metadata table from *Apache Iceberg: The Definitive Guide*:

| Metadata Table | What it shows |
|---|---|
| `history` | Snapshot lineage — every commit |
| `snapshots` | Snapshot details — files added/deleted, manifests |
| `files` | Current data files with stats |
| `manifests` | Manifest files that group data files |
| `partitions` | Partition-level stats |
| `refs` | Named references (branches / tags) |
| `metadata_log_entries` | Evolution of the metadata JSON files |

**Prerequisites:** Docker Compose stack must be running (`docker compose up -d`).

---
## 0 — Setup

In [2]:
import duckdb

con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("INSTALL iceberg")
con.execute("LOAD httpfs")
con.execute("LOAD iceberg")

# S3 credentials — DuckDB uses these to sign its own S3 requests directly
con.execute("""
CREATE OR REPLACE SECRET minio (
    TYPE        s3,
    KEY_ID      'minioadmin',
    SECRET      'minioadmin',
    ENDPOINT    'minio:9000',
    URL_STYLE   'path',
    USE_SSL     false,
    REGION      'us-east-1'
)
""")

# DuckDB 1.2+ requires OAuth2 credentials in a separate TYPE iceberg secret;
# SCOPE and other auth params are no longer accepted inline in ATTACH.
con.execute("""
CREATE OR REPLACE SECRET polaris_oauth (
    TYPE              iceberg,
    CLIENT_ID         'root',
    CLIENT_SECRET     's3cr3t',
    OAUTH2_SERVER_URI 'http://polaris:8181/api/catalog/v1/oauth/tokens',
    SCOPE             'PRINCIPAL_ROLE:ALL'
)
""")

# ACCESS_DELEGATION_MODE 'none' tells DuckDB to use the minio secret above
# rather than requesting vended credentials from Polaris.
con.execute("""
ATTACH 'demo_lh' AS polaris (
    TYPE                   iceberg,
    ENDPOINT               'http://polaris:8181/api/catalog',
    SECRET                 polaris_oauth,
    ACCESS_DELEGATION_MODE 'none'
)
""")

con.execute("SHOW ALL TABLES").df()

,database,schema,name,column_names,column_types,temporary
0,polaris,demo,transactions,[__],[UNKNOWN],False
1,polaris,demo,users,[__],[UNKNOWN],False


---
## 1 — Select a table

Choose which Iceberg table to inspect. All metadata sections below will query the selected table.

In [3]:
available_tables = [
    "polaris.demo.transactions",
    "polaris.demo.users",
]

table_selector = widgets.Dropdown(
    options=available_tables,
    value=available_tables[0],
    description="Table:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px"),
)

display(table_selector)

Dropdown(description='Table:', layout=Layout(width='400px'), options=('polaris.demo.transactions', 'polaris.de…

In [4]:
# Read the selected table name — re-run this cell after changing the dropdown
TABLE = table_selector.value
print(f"Inspecting: {TABLE}")

Inspecting: polaris.demo.transactions


---
## 2 — History table

The `history` metadata table tracks every snapshot that has ever been committed to the table — its lineage. Each row represents one commit.

**Schema**

| Column | Type | Description |
|---|---|---|
| `made_current_at` | timestamp | When this snapshot became the current one |
| `snapshot_id` | bigint | Unique snapshot identifier |
| `parent_id` | bigint | Parent snapshot ID (null for the first snapshot) |
| `is_current_ancestor` | boolean | Whether this snapshot is on the current lineage chain |

**Spark syntax:** `SELECT * FROM catalog.db.table.history`  
**Trino syntax:** `SELECT * FROM catalog.db."table$history"`  
**DuckDB (this notebook):** via `iceberg_history()` table function

In [ ]:
con.execute("SELECT * FROM polaris.demo.users ORDER BY created_at desc").df()

In [5]:
# Full snapshot history — most recent first
# DuckDB iceberg_snapshots() columns: sequence_number, snapshot_id, timestamp_ms, manifest_list
history_df = con.execute(f"""
    SELECT *
    FROM iceberg_snapshots('{TABLE}')
    ORDER BY timestamp_ms DESC
""").df()

print(f"Total snapshots: {len(history_df)}")
history_df

Total snapshots: 103


,sequence_number,snapshot_id,timestamp_ms,manifest_list
0,103,5887432978798942734,2026-05-15 18:48:09.665,s3://iceberg-warehouse/demo/transactions/metad...
1,102,6729638689664358371,2026-05-15 18:47:39.652,s3://iceberg-warehouse/demo/transactions/metad...
2,101,8044291316497407025,2026-05-15 18:47:09.687,s3://iceberg-warehouse/demo/transactions/metad...
3,100,4922461013160061509,2026-05-15 18:46:41.448,s3://iceberg-warehouse/demo/transactions/metad...
4,99,4997542972928376799,2026-05-15 18:46:09.636,s3://iceberg-warehouse/demo/transactions/metad...
...,...,...,...,...
98,5,5434184491940196438,2026-05-15 17:59:09.685,s3://iceberg-warehouse/demo/transactions/metad...
99,4,1339610066997163940,2026-05-15 17:58:39.612,s3://iceberg-warehouse/demo/transactions/metad...
100,3,4377751641787504921,2026-05-15 17:58:10.022,s3://iceberg-warehouse/demo/transactions/metad...
101,2,2420435701824803222,2026-05-15 17:57:39.608,s3://iceberg-warehouse/demo/transactions/metad...


In [6]:
# Interactive snapshot selector — pick any historical snapshot for time-travel
if history_df.empty:
    print("No history found.")
else:
    snapshot_options = [
        (f"{row.timestamp_ms}  →  snapshot {row.snapshot_id}", row.snapshot_id)
        for row in history_df.itertuples()
    ]

    snapshot_selector = widgets.Dropdown(
        options=snapshot_options,
        description="Snapshot:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="600px"),
    )
    display(snapshot_selector)

Dropdown(description='Snapshot:', layout=Layout(width='600px'), options=(('2026-05-15 18:48:09.665000  →  snap…

In [7]:
# Time-travel to the selected snapshot
# Re-run this cell after changing the dropdown above
selected_snapshot_id = snapshot_selector.value
print(f"Reading {TABLE} AS OF VERSION {selected_snapshot_id}")

time_travel_df = con.execute(f"""
    SELECT *
    FROM {TABLE}
    VERSION {selected_snapshot_id}
    LIMIT 20
""").df()

time_travel_df

Reading polaris.demo.transactions AS OF VERSION 5887432978798942734


ParserException: Parser Error: syntax error at or near "5887432978798942734"

LINE 4:     VERSION 5887432978798942734
                    ^

In [8]:
# Snapshot growth over time
# DuckDB iceberg_snapshots() does not expose operation or summary — only sequence_number, snapshot_id, timestamp_ms, manifest_list
con.execute(f"""
    SELECT
        sequence_number,
        snapshot_id,
        timestamp_ms,
        manifest_list
    FROM iceberg_snapshots('{TABLE}')
    ORDER BY timestamp_ms
""").df()

,sequence_number,snapshot_id,timestamp_ms,manifest_list
0,1,7911153381993429235,2026-05-15 17:57:13.090,s3://iceberg-warehouse/demo/transactions/metad...
1,2,2420435701824803222,2026-05-15 17:57:39.608,s3://iceberg-warehouse/demo/transactions/metad...
2,3,4377751641787504921,2026-05-15 17:58:10.022,s3://iceberg-warehouse/demo/transactions/metad...
3,4,1339610066997163940,2026-05-15 17:58:39.612,s3://iceberg-warehouse/demo/transactions/metad...
4,5,5434184491940196438,2026-05-15 17:59:09.685,s3://iceberg-warehouse/demo/transactions/metad...
...,...,...,...,...
99,100,4922461013160061509,2026-05-15 18:46:41.448,s3://iceberg-warehouse/demo/transactions/metad...
100,101,8044291316497407025,2026-05-15 18:47:09.687,s3://iceberg-warehouse/demo/transactions/metad...
101,102,6729638689664358371,2026-05-15 18:47:39.652,s3://iceberg-warehouse/demo/transactions/metad...
102,103,5887432978798942734,2026-05-15 18:48:09.665,s3://iceberg-warehouse/demo/transactions/metad...


---
## 3 — Snapshots table

While `history` shows lineage, the `snapshots` table reveals the **content** of each snapshot — how many manifest files, data files, and records were added or removed.

**Schema**

| Column | Type | Description |
|---|---|---|
| `committed_at` | timestamp | Wall-clock time of the commit |
| `snapshot_id` | bigint | Unique snapshot identifier |
| `parent_id` | bigint | Parent snapshot ID |
| `operation` | string | `append`, `overwrite`, `replace`, `delete` |
| `manifest_list` | string | Path to the manifest-list Avro file |
| `summary` | map<string,string> | Key metrics: added/deleted records, data files, manifests |

In [9]:
# All snapshots — iceberg_snapshots() columns: sequence_number, snapshot_id, timestamp_ms, manifest_list
con.execute(f"""
    SELECT *
    FROM iceberg_snapshots('{TABLE}')
    ORDER BY timestamp_ms DESC
""").df()

,sequence_number,snapshot_id,timestamp_ms,manifest_list
0,105,3558559139912783216,2026-05-15 18:49:09.653,s3://iceberg-warehouse/demo/transactions/metad...
1,104,8725004854740672672,2026-05-15 18:48:39.697,s3://iceberg-warehouse/demo/transactions/metad...
2,103,5887432978798942734,2026-05-15 18:48:09.665,s3://iceberg-warehouse/demo/transactions/metad...
3,102,6729638689664358371,2026-05-15 18:47:39.652,s3://iceberg-warehouse/demo/transactions/metad...
4,101,8044291316497407025,2026-05-15 18:47:09.687,s3://iceberg-warehouse/demo/transactions/metad...
...,...,...,...,...
100,5,5434184491940196438,2026-05-15 17:59:09.685,s3://iceberg-warehouse/demo/transactions/metad...
101,4,1339610066997163940,2026-05-15 17:58:39.612,s3://iceberg-warehouse/demo/transactions/metad...
102,3,4377751641787504921,2026-05-15 17:58:10.022,s3://iceberg-warehouse/demo/transactions/metad...
103,2,2420435701824803222,2026-05-15 17:57:39.608,s3://iceberg-warehouse/demo/transactions/metad...


In [27]:
# DuckDB does not expose operation/summary in iceberg_snapshots().
# Use iceberg_metadata() to get per-file stats per snapshot instead.
# iceberg_metadata() columns: manifest_path, manifest_sequence_number, manifest_content,
#                              status, content, file_path, file_format, record_count
con.execute(f"""
    SELECT
        manifest_sequence_number,
        manifest_content,
        status,
        content,
        file_format,
        count(*)            AS file_count,
        sum(record_count)   AS total_records
    FROM iceberg_metadata('{TABLE}')
    GROUP BY ALL
    ORDER BY manifest_sequence_number DESC
""").df()

,manifest_sequence_number,manifest_content,status,content,file_format,file_count,total_records
0,125,DATA,ADDED,EXISTING,PARQUET,1,59.0
1,124,DATA,ADDED,EXISTING,PARQUET,1,60.0
2,123,DATA,ADDED,EXISTING,PARQUET,1,59.0
3,122,DATA,ADDED,EXISTING,PARQUET,1,60.0
4,121,DATA,ADDED,EXISTING,PARQUET,1,60.0
5,120,DATA,ADDED,EXISTING,PARQUET,1,59.0
6,119,DATA,ADDED,EXISTING,PARQUET,1,60.0
7,118,DATA,ADDED,EXISTING,PARQUET,1,59.0
8,117,DATA,ADDED,EXISTING,PARQUET,1,60.0
9,116,DATA,ADDED,EXISTING,PARQUET,1,59.0


---
## 4 — Files table

The `files` metadata table lists every **current data file** tracked by the latest snapshot, together with per-file column-level statistics (null counts, lower/upper bounds). This is the key table for diagnosing data skew, small-file problems, and partition layout.

**Schema (key columns)**

| Column | Type | Description |
|---|---|---|
| `content` | int | 0 = DATA, 1 = POSITION_DELETES, 2 = EQUALITY_DELETES |
| `file_path` | string | Full S3/HDFS path |
| `file_format` | string | `PARQUET`, `ORC`, `AVRO` |
| `record_count` | bigint | Rows in this file |
| `file_size_in_bytes` | bigint | Compressed file size |
| `column_sizes` | map<int,bigint> | Bytes used per column field ID |
| `value_counts` | map<int,bigint> | Non-null values per field ID |
| `null_value_counts` | map<int,bigint> | Null values per field ID |
| `lower_bounds` | map<int,bytes> | Min value per field ID |
| `upper_bounds` | map<int,bytes> | Max value per field ID |

In [10]:
con.execute(f"""
    SELECT
        *
    FROM iceberg_metadata('{TABLE}')
    WHERE status != 'DELETED'
    ORDER BY record_count DESC
""").df()

,manifest_path,manifest_sequence_number,manifest_content,status,content,file_path,file_format,record_count
0,s3://iceberg-warehouse/demo/transactions/metad...,100,DATA,EXISTING,EXISTING,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,44189
1,s3://iceberg-warehouse/demo/transactions/metad...,100,DATA,EXISTING,EXISTING,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,60
2,s3://iceberg-warehouse/demo/transactions/metad...,100,DATA,EXISTING,EXISTING,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,60
3,s3://iceberg-warehouse/demo/transactions/metad...,103,DATA,ADDED,EXISTING,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,60
4,s3://iceberg-warehouse/demo/transactions/metad...,102,DATA,ADDED,EXISTING,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,60
...,...,...,...,...,...,...,...,...
101,s3://iceberg-warehouse/demo/transactions/metad...,100,DATA,EXISTING,EXISTING,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,59
102,s3://iceberg-warehouse/demo/transactions/metad...,100,DATA,EXISTING,EXISTING,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,59
103,s3://iceberg-warehouse/demo/transactions/metad...,100,DATA,EXISTING,EXISTING,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,59
104,s3://iceberg-warehouse/demo/transactions/metad...,100,DATA,EXISTING,EXISTING,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,58


In [11]:
# Current data files via iceberg_metadata()
# iceberg_metadata() columns: manifest_path, manifest_sequence_number, manifest_content,
#                              status, content, file_path, file_format, record_count
files_df = con.execute(f"""
    SELECT
        content,
        file_path,
        file_format,
        record_count
    FROM iceberg_metadata('{TABLE}')
    WHERE status != 'DELETED'
    ORDER BY record_count DESC
""").df()

print(f"Total data files: {len(files_df)}")
files_df

Total data files: 106


,content,file_path,file_format,record_count
0,EXISTING,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,44189
1,EXISTING,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,60
2,EXISTING,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,60
3,EXISTING,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,60
4,EXISTING,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,60
...,...,...,...,...
101,EXISTING,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,59
102,EXISTING,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,59
103,EXISTING,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,59
104,EXISTING,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,58


In [12]:
# File stats — iceberg_metadata() does not expose file_size_in_bytes
# We can report record counts per file
con.execute(f"""
    SELECT
        count(*)                        AS file_count,
        sum(record_count)               AS total_records,
        round(avg(record_count), 0)     AS avg_records_per_file,
        min(record_count)               AS min_records,
        max(record_count)               AS max_records
    FROM iceberg_metadata('{TABLE}')
    WHERE status != 'DELETED'
      AND content = 'DATA'
""").df()

,file_count,total_records,avg_records_per_file,min_records,max_records
0,0,NaN,NaN,<NA>,<NA>


---
## 5 — Manifests table

Manifest files are Avro files that list groups of data files. The `manifests` table exposes every manifest file referenced by the **current snapshot**, including partition-level stats that let Iceberg prune manifests at planning time — without opening a single data file.

**Schema (key columns)**

| Column | Type | Description |
|---|---|---|
| `path` | string | S3/HDFS path to the Avro manifest file |
| `length` | bigint | Manifest file size in bytes |
| `partition_spec_id` | int | Which partition spec this manifest was written under |
| `added_snapshot_id` | bigint | Snapshot that added this manifest |
| `added_data_files_count` | int | Data files added in this manifest |
| `existing_data_files_count` | int | Data files not modified since last compaction |
| `deleted_data_files_count` | int | Logically deleted data files pending cleanup |
| `added_rows_count` | bigint | Rows added across all files in this manifest |
| `existing_rows_count` | bigint | Unchanged rows |
| `deleted_rows_count` | bigint | Deleted rows not yet compacted away |

In [13]:
# Manifests via iceberg_metadata() — grouped by manifest file
con.execute(f"""
    SELECT
        manifest_path,
        manifest_sequence_number,
        manifest_content,
        count(*)            AS file_count,
        sum(record_count)   AS total_records,
        count(CASE WHEN status = 'DELETED' THEN 1 END) AS deleted_files
    FROM iceberg_metadata('{TABLE}')
    GROUP BY manifest_path, manifest_sequence_number, manifest_content
    ORDER BY manifest_sequence_number DESC
""").df()

,manifest_path,manifest_sequence_number,manifest_content,file_count,total_records,deleted_files
0,s3://iceberg-warehouse/demo/transactions/metad...,106,DATA,1,60.0,0
1,s3://iceberg-warehouse/demo/transactions/metad...,105,DATA,1,60.0,0
2,s3://iceberg-warehouse/demo/transactions/metad...,104,DATA,1,59.0,0
3,s3://iceberg-warehouse/demo/transactions/metad...,103,DATA,1,60.0,0
4,s3://iceberg-warehouse/demo/transactions/metad...,102,DATA,1,60.0,0
5,s3://iceberg-warehouse/demo/transactions/metad...,101,DATA,1,59.0,0
6,s3://iceberg-warehouse/demo/transactions/metad...,100,DATA,100,50091.0,0


In [14]:
# Manifest health — pending deletes signal compaction is overdue
con.execute(f"""
    SELECT
        count(DISTINCT manifest_path)                       AS manifest_count,
        count(CASE WHEN status != 'DELETED' THEN 1 END)    AS active_files,
        count(CASE WHEN status = 'DELETED' THEN 1 END)     AS pending_deleted_files,
        sum(CASE WHEN status != 'DELETED' THEN record_count ELSE 0 END) AS active_records,
        sum(CASE WHEN status = 'DELETED' THEN record_count ELSE 0 END)  AS pending_deleted_records
    FROM iceberg_metadata('{TABLE}')
""").df()

,manifest_count,active_files,pending_deleted_files,active_records,pending_deleted_records
0,7,106,0,50449.0,0.0


---
## 6 — Partitions table

The `partitions` metadata table provides a **partition-level summary** of the current snapshot — record counts, file counts, and column statistics rolled up per partition value. This is essential for spotting data skew.

**Schema**

| Column | Type | Description |
|---|---|---|
| `partition` | struct | Partition values (depends on the table's partition spec) |
| `record_count` | bigint | Total rows in this partition |
| `file_count` | int | Number of data files in this partition |
| `total_size` | bigint | Uncompressed bytes |
| `spec_id` | int | Partition spec ID used to write these files |

> **Note:** If the table is unpartitioned, this table will show a single row.

In [15]:
# DuckDB has no iceberg_partitions() — approximate partition distribution via iceberg_scan()
# by reading the actual data and grouping on the partition column.
# Adjust the GROUP BY column to match your table's partition spec.
con.execute(f"""
    SELECT
        date_trunc('day', transaction_date) AS partition_day,
        count(*)                            AS record_count
    FROM iceberg_scan('{TABLE}')
    GROUP BY 1
    ORDER BY 1
""").df()

BinderException: Binder Error: Referenced column "transaction_date" not found in FROM clause!
Candidate bindings: "transaction_id", "amount", "status", "event_time", "user_id"

LINE 3:         date_trunc('day', transaction_date) AS partition_day,
                                  ^

In [34]:
# Skew detection — reuse the partition query above
part_df = con.execute(f"""
    SELECT
        count(*)                            AS partition_count,
        min(record_count)                   AS min_records,
        max(record_count)                   AS max_records,
        round(avg(record_count), 0)         AS avg_records,
        round(stddev(record_count), 0)      AS stddev_records
    FROM (
        SELECT
            date_trunc('day', transaction_date) AS partition_day,
            count(*)                            AS record_count
        FROM iceberg_scan('{TABLE}')
        GROUP BY 1
    )
""").df()

part_df

BinderException: Binder Error: Referenced column "transaction_date" not found in FROM clause!
Candidate bindings: "transaction_id", "amount", "status", "event_time", "user_id"

LINE 10:             date_trunc('day', transaction_date) AS partition_day,
                                       ^

---
## 7 — Refs table (branches & tags)

Iceberg's standard `refs` metadata table shows all named references — branches and tags — in the catalog for this table. With Apache Polaris only the default `main` branch exists.

> **Note:** DuckDB's Iceberg extension does not expose a `refs` metadata function. Use Trino's `"table$refs"` instead (see `trino_iceberg_metadata.ipynb`).

---
## 8 — Metadata log entries

Each time a snapshot is committed, Iceberg writes a new `metadata.json` file and appends the path to the **metadata log**. The `metadata_log_entries` table exposes this log — it is the record of how the table's metadata has evolved over time.

**Schema**

| Column | Type | Description |
|---|---|---|
| `timestamp` | timestamp | When this metadata file was created |
| `file` | string | Path to the `metadata.json` file |
| `latest_snapshot_id` | bigint | Snapshot captured in that metadata file |
| `latest_schema_id` | int | Schema version at that point |
| `latest_sequence_number` | bigint | Monotonically increasing write counter |

---
## 9 — Table health dashboard

Combine all metadata sources into a single health summary — the kind of thing you'd check daily to decide whether compaction, expiry, or repartitioning is needed.

In [38]:
from datetime import timezone

# ── Snapshot count ──────────────────────────────────────────────────────────
snapshot_count = con.execute(f"SELECT count(*) FROM iceberg_snapshots('{TABLE}')").fetchone()[0]

# ── File stats (via iceberg_metadata) ──────────────────────────────────────
file_stats = con.execute(f"""
    SELECT
        count(*)                    AS file_count,
        sum(record_count)           AS total_records,
        round(avg(record_count), 0) AS avg_records_per_file
    FROM iceberg_metadata('{TABLE}')
    WHERE status != 'DELETED'
      AND content = 'DATA'
""").df().iloc[0]

# ── Manifest bloat ──────────────────────────────────────────────────────────
manifest_stats = con.execute(f"""
    SELECT
        count(DISTINCT manifest_path)                                       AS manifest_count,
        count(CASE WHEN status = 'DELETED' THEN 1 END)                     AS pending_deleted_files,
        sum(CASE WHEN status = 'DELETED' THEN record_count ELSE 0 END)     AS pending_deleted_records
    FROM iceberg_metadata('{TABLE}')
""").df().iloc[0]

# ── Print ───────────────────────────────────────────────────────────────────
print(f"{'='*52}")
print(f"  TABLE HEALTH: {TABLE}")
print(f"{'='*52}")
print(f"  Snapshots committed      : {snapshot_count}")
print(f"  Current data files       : {int(file_stats.file_count)}")
print(f"  Total records            : {int(file_stats.total_records):,}")
print(f"  Avg records per file     : {int(file_stats.avg_records_per_file):,}")
print(f"  Manifests                : {int(manifest_stats.manifest_count)}")
print(f"  Pending deleted files    : {int(manifest_stats.pending_deleted_files)}")
print(f"  Pending deleted records  : {int(manifest_stats.pending_deleted_records):,}")

# ── Recommendations ─────────────────────────────────────────────────────────
print()
print("  Recommendations:")
if file_stats.avg_records_per_file < 10_000:
    print("  ⚠  Low records-per-file — consider running compaction (RewriteDataFiles)")
else:
    print("  ✓  File sizes look healthy")
if manifest_stats.pending_deleted_files > 0:
    print("  ⚠  Pending deletes in manifests — run expireSnapshots to reclaim space")
else:
    print("  ✓  No pending deletes")

print(f"{'='*52}")

  TABLE HEALTH: polaris.demo.transactions
  Snapshots committed      : 1
  Current data files       : 0


ValueError: cannot convert float NaN to integer